# 10 · Sparse autoencoders, and removing superposition with L1

**Paper:** "Three Ways Out": Approach 1, creating models without superposition (p. 42), and
Approach 2, finding an overcomplete basis (pp. 42–44, footnote 20).
**Experiments:** E10 `sae_recovery`, E11 `l1_removes_superposition`.
**Library:** `superposition.sae`.

**Learning goals**

1. Write down a sparse autoencoder (SAE) and say what each piece is for: the ReLU encoder, the
   unit-norm decoder, the L1 penalty, TopK, dead-latent resampling.
2. Score an SAE against the *true* features, which only a toy model makes possible (p. 44).
3. See that low reconstruction error does not mean the features were found.
4. Remove superposition during training instead, with an L1 penalty on h, and read the price
   off the loss (p. 42).

The quick profiles take about 7 minutes each on a laptop CPU.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition import lab, sae
from superposition.data import IndependentSparse
from superposition.experiments.runner import run_experiment
from superposition.plotting.sae import dictionary_2d_figure
from superposition.runs import list_runs
from superposition.utils import make_generator

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    # Your latest complete quick-profile run of `name` with its figures, or a fresh one. Runs
    # made without figures (`superposition run --no-figures`, or the replication tests) are
    # skipped: this notebook shows the run's figures.
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]

## 1. An SAE on the pentagon, by hand

First a toy model in superposition: the p. 2 pentagon (n = 5 features in m = 2 dimensions,
1 − S = 0.1). Its hidden activations are h = Wx, points in the plane.

An SAE rewrites each h as a sparse combination of learned directions:

    f = ReLU(W_enc (h − b_dec) + b_enc),   ĥ = W_dec f + b_dec,   loss = ‖h − ĥ‖² + λ‖f‖₁

The columns of W_dec (kept at unit norm) are the SAE's guesses for the feature directions.

**Predict:** with exactly 5 latents, can the SAE find the 5 pentagon directions? With 20?

In [ ]:
toy = lab.train_playground(lab.PlaygroundConfig())  # the pentagon, best of 4 restarts
W = torch.as_tensor(toy.best_W)
dist = IndependentSparse(0.1, 1, 5)


def sample(batch, generator):
    return dist.sample(batch, generator)[:, 0] @ W.T


sizes = [5, 20]
mask = torch.zeros(2, 20, dtype=torch.bool)
mask[0, :5] = True
mask[1, :] = True
model = sae.SparseAutoencoder(2, 20, 2, l1=0.3, resample=True, latent_mask=mask)
result = sae.train_sae(model, sample, sae.SAETrainConfig(steps=4000, batch_size=1024))
h = sample(50_000, make_generator(1))
stats = sae.sae_stats(model, h)
represented = torch.linalg.vector_norm(W, dim=0) > 0.5
rec = sae.recovery(model.unit_decoder(), stats.alive, W, represented)
pd.DataFrame(
    {
        "latents": sizes,
        "recovered (cos > 0.95)": rec.fraction[0.95].numpy(),
        "mean max cos": rec.mean_max_cos.numpy(),
        "L0": stats.l0.numpy(),
        "FVU": stats.fvu.numpy(),
        "dead": stats.dead_fraction.numpy(),
    }
)

In [ ]:
panels = [
    (
        f"{s} latents",
        model.unit_decoder()[k].numpy(),
        mask[k].numpy(),
        stats.alive[k].numpy(),
        float(rec.fraction[0.95][k]),
    )
    for k, s in enumerate(sizes)
]
dictionary_2d_figure(toy.best_W, panels, 0.95)

Every SAE instance above trained in the same model: like the toy models, the SAE has a leading
instance dimension, and `latent_mask` lets different dictionary sizes share it.

## 2. The sweep (E10)

E10 does this for the pentagon, the sparse n = 20, m = 5 models of p. 11, and one n = 400,
m = 30 model from the uniform-superposition setup (p. 17), over dictionary size × λ ×
resampling, plus TopK SAEs. The best SAE of each sweep is picked **by ground truth**, so the
run also reports how many settings pass.

In [ ]:
run10 = quick_run("sae_recovery")
criteria(run10)

In [ ]:
display(Image(filename=str(run10.figures_dir / "recovery_sweep.png")))
display(Image(filename=str(run10.figures_dir / "pentagon_dictionary.png")))

**Reconstruction is not recovery.** Each dot below is one SAE. Many reconstruct h almost
perfectly (low fraction of variance unexplained) without finding the features: an SAE can
reconstruct the plane with a dense code (four latents ±e₁, ±e₂ reconstruct any 2D point
exactly). Look at the n = 400 panel: which TopK SAE reconstructs better, and which one finds
the features?

In [ ]:
display(Image(filename=str(run10.figures_dir / "reconstruction_vs_recovery.png")))
table = run10.read_table("metrics")
table.loc[
    table["toy_model"] == "n400_m30_p0.1",
    ["size", "l1", "topk", "recovered_0.9", "mean_max_cos", "l0", "fvu", "true_l0"],
]

## 3. Removing superposition during training (E11)

Approach 1 (p. 42): add λ‖h‖₁ to the loss. For the ReLU output model h = Wx can be negative,
and the penalty charges |h|, which is cheapest when features point along the hidden basis
(‖Wᵢ‖₁ = ‖Wᵢ‖₂ only for basis-aligned Wᵢ). So the penalty makes the hidden basis privileged,
and "kills features below a certain importance threshold, especially if they're not basis
aligned".

**Predict:** as λ grows, which pentagon features die first? Will the last two survivors be
orthogonal, or an antipodal pair? (Hint: does the L1 penalty charge an antipodal pair along
e₁ more than two features along e₁ and e₂?)

In [ ]:
run11 = quick_run("l1_removes_superposition")
criteria(run11)

In [ ]:
display(Image(filename=str(run11.figures_dir / "l1_sweep.png")))
display(Image(filename=str(run11.figures_dir / "pentagon_arrows.png")))
run11.read_table("metrics")[
    [
        "setup",
        "lambda",
        "loss",
        "represented",
        "superposed",
        "formal_superposition",
        "dims_per_feature",
    ]
]

The loss column is the reconstruction loss *without* the penalty, so the rows are comparable:
they are all the same task (footnote 17 warns against comparing losses across sparsities, not
across λ). Read the price of removing superposition off it.

## Exercises

**1.** Why must the decoder columns be kept at unit norm?

<details><summary>Solution</summary>

The L1 penalty is on f. Without a norm constraint the SAE could halve every f and double every
decoder column: same reconstruction, half the penalty. Repeating that drives the penalty to 0
and the constraint away. Fixing ‖W_dec[:, j]‖ = 1 removes that direction of freedom.
</details>

**2.** `sae.recovery` uses the *signed* cosine. Why would |cos| be wrong here?

<details><summary>Solution</summary>

SAE codes are non-negative, so a latent can only add +d_j. A latent pointing along −Wᵢ never
represents feature i, whose values are non-negative (x ∈ [0, 1]); it may represent another
feature that happens to sit opposite (an antipodal partner).
</details>

**3.** Use `sae.sae_stats` on the SAEs from section 1 to count dead latents, then retrain with
`resample=False`. Does resampling change recovery here? Look at `result.resampled`.

<details><summary>Solution</summary>

```python
off = sae.SparseAutoencoder(2, 20, 2, l1=0.3, resample=False, latent_mask=mask)
sae.train_sae(off, sample, sae.SAETrainConfig(steps=4000, batch_size=1024))
print(result.resampled, sae.sae_stats(off, h).dead_fraction)
```

A latent is resampled only if it fired on fewer than 1 in 100,000 samples during a check
window (D-503), so with dense enough data few latents qualify and the two runs differ mostly
through their initialization. Compare E10's `resampling` row.
</details>

**4.** In E11 the reconstruction loss at the largest λ equals the loss of predicting the mean
for every feature. Compute that value for the pentagon from p/3 − p²/4 and the importances.

<details><summary>Solution</summary>

Σᵢ 0.9ⁱ (p/3 − p²/4) with p = 0.1: (1 + 0.9 + 0.81 + 0.729 + 0.6561) · (0.03333 − 0.0025)
≈ 4.095 · 0.03083 ≈ 0.126.
</details>

**5.** Footnote 21 argues that a little L1 should reduce superposition "with essentially no
effect on performance". Where in the E11 table do you see that margin?

<details><summary>Solution</summary>

At the smallest λ values the loss barely moves while D* rises (fewer features per dimension)
for the n = 20 model: the model gives up its least useful superposed features almost for free,
because at its optimum the derivative of the loss with respect to the amount of superposition
is zero.
</details>